In [1]:
import pandas as pd

from src.grid import GridEnvironment
from src.search import (
    bfs,
    greedy_best_first_search,
    a_star_search,
    uniform_cost_search,
)
from src.heuristics import (
    manhattan_distance,
)
from src.metrics import (
    measure_execution_time,
    benchmark_algorithm,
    summarize_results
)
from src.visualization import (
    print_grid,
    plot_grid,
    print_terrain,
)
from src.experiments import (
    generate_solvable_grid,
    run_single_experiment,
    run_configured_experiment,
)
from src.config import GridExperimentConfig


c:\Users\niush\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
rows = 30
cols = 30

start = (0,0)
goal = (29,29)

obstacle_probability = 0.2

In [3]:
grid = generate_solvable_grid(
    rows,
    cols,
    obstacle_probability,
    start,
    goal,
    bfs,
)

In [4]:
algorithms = [
    (
        "BFS",
        bfs,
        {}
    ),

    (
        "UCS",
        uniform_cost_search,
        {}
    ),

    (
        "Greedy",
        greedy_best_first_search,
        {
            "heuristic": manhattan_distance
        }
    ),

    (
        "A*",
        a_star_search,
        {
            "heuristic": manhattan_distance
        }
    ),
]

In [5]:
results = run_single_experiment(
    environment=grid,
    start=start,
    goal=goal,
    algorithms=algorithms,
)

In [6]:
df = pd.DataFrame(results)

df

,Algorithm,Success,Path Length,Path Cost,Nodes Expanded,Nodes Generated,Max Frontier,Execution Time
0,BFS,True,58,58.0,722,722,33,0.006381
1,UCS,True,58,58.0,722,722,33,0.005504
2,Greedy,True,60,60.0,72,128,57,0.000705
3,A*,True,58,58.0,383,419,46,0.003943


On small unweighted grids, BFS and UCS show identical behavior because all edge costs are equal and the search space is limited. Larger randomized environments are used for meaningful scalability analysis.

### Experiment Result Summary

On a 30×30 unweighted grid with 20% obstacles, BFS and UCS produced the same optimal path because all transition costs were equal. A* achieved the same optimal solution while expanding fewer nodes by leveraging the Manhattan heuristic. Greedy Best-First Search explored significantly fewer nodes but returned a longer, non-optimal path, highlighting the trade-off between search efficiency and optimality.

BFS:
Shortest path

UCS:
Cheapest path

A*:
Cheapest path with fewer expansions

Defining Terrain

In [7]:
terrain_costs = {
    (0,1): 1,
    (0,2): 9,
    (0,3): 9,
    (1,0): 1,
    (1,1): 1,
    (1,2): 1,
    (1,3): 1,
}

In [8]:
weighted_grid = GridEnvironment(
    rows=2,
    cols=5,
    terrain_costs=terrain_costs,
)

start = (0,0)
goal = (0,4)

In [9]:
bfs_result = bfs(
    weighted_grid,
    start,
    goal,
)


ucs_result = uniform_cost_search(
    weighted_grid,
    start,
    goal,
)


greedy_result = greedy_best_first_search(
    weighted_grid,
    start,
    goal,
    manhattan_distance,
)


astar_result = a_star_search(
    weighted_grid,
    start,
    goal,
    manhattan_distance,
)

In [10]:
results = {
    "BFS": bfs_result,
    "UCS": ucs_result,
    "Greedy": greedy_result,
    "A*": astar_result,
}


for name, result in results.items():

    print(name)

    print(
        "Path:",
        result.path
    )

    print(
        "Depth:",
        result.solution_depth
    )

    print(
        "Cost:",
        result.path_cost
    )

    print()

BFS
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
Depth: 4
Cost: 20.0

UCS
Path: [(0, 0), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (0, 4)]
Depth: 6
Cost: 6.0

Greedy
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
Depth: 4
Cost: 20.0

A*
Path: [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3), (1, 4), (0, 4)]
Depth: 6
Cost: 6.0



In [11]:
print_terrain(weighted_grid)

1.0 1 9 9 1.0
1 1 1 1 1.0


### Weighted Grid Experiment Summary

The weighted grid experiment highlights the difference between minimizing path length and minimizing path cost. BFS and Greedy Best-First Search selected the shorter geometric route, resulting in a higher total cost. UCS and A* successfully avoided expensive cells and found a lower-cost solution. This demonstrates why cost-aware algorithms are required for weighted environments.

**Making Config**

In [12]:
config = GridExperimentConfig(
    rows=30,
    cols=30,
    obstacle_probability=0.2,
    start=(0,0),
    goal=(29,29),
    trials=5,
)

In [13]:
results = run_configured_experiment(
    config,
    algorithms,
    bfs,
)

In [14]:
df = pd.DataFrame(results)

df

,Algorithm,Success,Path Length,Path Cost,Nodes Expanded,Nodes Generated,Max Frontier,Execution Time,Trial,Grid Size,Obstacle Probability
0,BFS,True,58,58.0,715,715,30,0.003404,0,30x30,0.2
1,UCS,True,58,58.0,715,715,30,0.004342,0,30x30,0.2
2,Greedy,True,62,62.0,72,133,62,0.000636,0,30x30,0.2
3,A*,True,58,58.0,502,563,70,0.003219,0,30x30,0.2
4,BFS,True,58,58.0,729,729,30,0.003562,1,30x30,0.2
5,UCS,True,58,58.0,729,729,30,0.004541,1,30x30,0.2
6,Greedy,True,72,72.0,84,147,64,0.000634,1,30x30,0.2
7,A*,True,58,58.0,539,597,70,0.003574,1,30x30,0.2
8,BFS,True,58,58.0,731,731,32,0.003534,2,30x30,0.2
9,UCS,True,58,58.0,731,731,32,0.004513,2,30x30,0.2


In [15]:
summary = (
    df
    .groupby("Algorithm")
    [
        [
            "Path Cost",
            "Nodes Expanded",
            "Execution Time",
        ]
    ]
    .mean()
)

summary

,Path Cost,Nodes Expanded,Execution Time
Algorithm,,,
A*,58.0,535.4,0.003451
BFS,58.0,725.8,0.003547
Greedy,66.8,85.6,0.000632
UCS,58.0,725.8,0.004368


In [16]:
summary = summarize_results(df)

summary

,Algorithm,Success,Path Cost,Nodes Expanded,Nodes Generated,Execution Time
0,A*,1.0,58.0,535.4,591.0,0.003451
1,BFS,1.0,58.0,725.8,725.8,0.003547
2,Greedy,1.0,66.8,85.6,143.2,0.000632
3,UCS,1.0,58.0,725.8,725.8,0.004368
